In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Entropy and the Boltzmann Distribution

Recall _entropy_, the measure of the number of microstates (multiplicity) about a system given its macroscopic properties (e.g., temperature, pressure, volume).  It measures how probability is spread over the possible _microstates_ of the system.

$$
S = -k\sum_\omega p_\omega \log p_\omega,
$$

where $\omega$ enumerates all possible microstates, and $p_\omega$ is the probability of the system occupying each state.  This is the _statistical mechanics_ definition of entropy.  The _thermodynamic_ definition of entropy $dS = \frac{\delta Q}{T}$ relating the change in entropy in a closed system to incremental and reversable transfer of heat $\delta Q$ into that system of temperature $T$.  Demanding these definitions be equal (and making use of the _fundamental thermodynamic relation_ $dU = T dS - p dV$) leads to the _Boltzmann distribution_ as the solution for describing the probability of a system occupying a particular microstate $\omega$ is

$$
p_\omega = \frac{e^{-E_\omega/kT}}{Z},
$$

where $Z$ the normalization constant resulting from summing the numerator over all possible microstates of the system $\Omega$

$$
Z = \sum_{\omega'\in \Omega}e^{-E_{\omega'}/kT}.
$$

## The Ising Model

We'll use the Boltzmann distribution above to simulate a ferromagnet using a simple, two-dimensional _Ising model_. This model constructs a ferromagnet as a 2-D lattice with $N$ sites, and the spin of each lattice site can be up or down, which we'll representent as $+1$ and $-1$, respectively.

A configuration of the microstate of this system is given by $\omega = \{\omega_1, \omega_2, \dots, \omega_N\}$, where $\omega_i=\pm 1$, meaning there are $|\Omega|=2^N$ possible microstates of the system.

Let's construct a small latice with random initial spins.

We'll define the energy of a particular configuration to be

$$
E_\omega = -J\sum_{\left<i,j\right>}\omega_i\omega_j - H\sum_{i=1}^N\omega_i,
$$

where $\left<i,j\right>$ indicates adjacent (i.e., nearest neighbor) sites (vertically or horizontally), $J>0$ drives the strength of coupling between nearest-neighbors, and $H>0$ represents the strength of an external field.

Now that we can calculate the energy of any particular microstate $\omega$, we can calculate the (relative) probability of occupying that state using the Boltzmann distribution.

I say relative here because we haven't gone through the effort of summing the numerator of the Boltzmann distribution over all possible microstates (i.e., we haven't normalized the distribution).  With this we can calculate the _relative_ probability of the system occupying two different microstates.

## Phase Transition

We can use our model to explore phase transitions, particularly in the _magnetization_. The _magnetization_ of our model for a particular microstate $\omega$ is

$$
M_\omega = \sum_{i=1}^N \omega_i.
$$

To determine the macroscopic magnetization at temperature $T$ we need to calculate the expectation value of this magnetization over possible microstates $\left<M\right>_T$.

$$
\left<M\right>_T = \sum_{\omega \in \Omega} M_\omega p(\omega) = \sum_{\omega \in \Omega} M_\omega \mathrm{Boltz(\omega)}.
$$

Importance sampling to the rescue!  Recall that using some sampling distribution $g(\omega)$ we can estimate such an expectation value using $S$ draws $\omega^1, \dots, \omega^S$ from $g(\omega)$ as

$$
E(h(\omega)) = \frac{\sum_{s=1}^S h(\omega^s) w(\omega^s)}{\sum_{s=1}^S w(\omega^s)}
$$

where

$$
w(\omega^s) = \frac{\mathrm{Boltz}(\omega^s)}{g(\omega^s)}
$$

are the _importance weights_.

Let's use our `draw_state()` function as our sampling distribution.  Since it's uniform over each lattice point's possible spin value $\omega_i$ we don't need to worry about explicitly calculating it for our importance weights (there's just a normalization constant that cancels).  To avoid overflow/underflow we're going to stick to working in the (natural) log as much as possible.

To avoid having to explicitly exponentiate each log term, sum over them, then take the log again (a process very prone to over/underflow, we can make use of some mathematical tricks with `logsumexp()`.

We can use this fuction to calculate, for example, the log of the sum of the weights in the following way.

For the numerator of our importance sampling we need to multiply each weight by the magnetization, which we can do with the `b=` keyword argument.  This should be everything we need.

...but weight.  We should look at them. 

From Chap. 10 of Gelman, we know that we should look at the distribution of weights to get a sense of the accuracy of our estimate.  If there are relatively few extreme weights, then their contributions will dominate our estimate above.  To be more quantitative, we can estimate the effective sample size (ESS).  If this is $\gg1$ we're in good shape, and if it's ${\sim}1$, not so good....